In [6]:
import os
from datasets import load_dataset

HF_TOKEN = os.getenv("HF_TOKEN")

dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="test",
    streaming=True,
    token=HF_TOKEN
)

sample = next(iter(dataset))

print(sample["id"])
print(sample["audio"])

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

6930-75918-0000


In [7]:
audio = sample["audio"]

print(audio)
print(audio.get_all_samples())

AudioSamples:
  data (shape): torch.Size([1, 56080])
  pts_seconds: 0.0
  duration_seconds: 3.505
  sample_rate: 16000



In [8]:
import soundfile as sf

audio = sample["audio"].get_all_samples()

audio_array = audio.data.squeeze(0).numpy()

sf.write(
    "test_audio.wav",
    audio_array,
    audio.sample_rate
)

print("Saved: test_audio.wav")

Saved: test_audio.wav


In [1]:
from faster_whisper import WhisperModel
import hashlib
import re

def calculate_audio_hash(audio_path: str) -> str:
    sha256 = hashlib.sha256()
    with open(audio_path, "rb") as audio_file:
        for chunk in iter(lambda: audio_file.read(1024 * 1024), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

def deduplicate_audio(audio_path: str, processed_hashes: set) -> bool:
    """
    Returns True if new | False if the audio was already processed.
    """
    audio_hash = calculate_audio_hash(audio_path)
    if audio_hash in processed_hashes:
        print("Duplicate audio detected.")
        return False
    processed_hashes.add(audio_hash)
    print(f"New audio accepted. Hash: {audio_hash[:12]}...")
    return True

def ai_governance_check(audio_path: str) -> bool:
    """
    Governance layer.
    - Authentication / authorization
    - Data classification
    - Tenant isolation
    - Audit logging
    - Retention policy
    - Model/version tracking
    - PII handling
    """
    allowed_extensions = {".wav", ".mp3", ".mpeg", ".flac", ".m4a"}
    extension = "." + audio_path.split(".")[-1].lower()
    if extension not in allowed_extensions:
        print("Governance reject unsupported audio format.")
        return False
    print("transcription request accepted")
    print("model=whisper-large-v3")

    return True

def transcription_guardrail(text: str) -> str:
    """
    quality guardrail.
    - Toxic-content detection
    - Prompt-injection detection
    - Sensitive-data filtering
    - Length limits
    - Output validation
    """

    if not text or not text.strip():
        raise ValueError("Empty transcription returned.")
    text = re.sub(
        r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
        "[EMAIL_REDACTED]",
        text
    )
    text = re.sub(
        r"\b\d{10}\b",
        "[PHONE_REDACTED]",
        text
    )
    #func to remove any other sensitive information can be added here
    return text.strip()

def transcribe_audio(audio_path: str, model: WhisperModel) -> list:
    """
    Transcribe audio and return timestamped segments.
    """
    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        vad_filter=True
    )

    results = []
    for segment in segments:
        text = transcription_guardrail(segment.text)
        results.append({
            "start": round(segment.start, 2),
            "end": round(segment.end, 2),
            "text": text
        })
    return results


In [2]:
def transcription_pipeline(audio_path: str):

    # my first preference Redis / DynamoDB / PostgreSQL
    processed_hashes = set()

    if not deduplicate_audio(audio_path, processed_hashes):
        return {
            "status": "duplicate",
            "message": "Audio has already been processed."
        }

    if not ai_governance_check(audio_path):
        return {
            "status": "rejected",
            "message": "Audio failed governance validation."
        }

    model = WhisperModel(
        "large-v3",
        device="cpu",
        compute_type="int8"
    )
    segments = transcribe_audio(
        audio_path,
        model
    )

    return {
        "status": "success",
        "model": "whisper-large-v3",
        "segments": segments
    }

if __name__ == "__main__":

    result = transcription_pipeline(
        "test_audio.wav"
    )

    print("\nFinal Result:")
    print(result)

New audio accepted. Hash: 103c3f15eb37...
transcription request accepted
model=whisper-large-v3

Final Result:
{'status': 'success', 'model': 'whisper-large-v3', 'segments': [{'start': 0.18, 'end': 3.5, 'text': 'concord returned to its place amidst the tents'}]}


In [ ]:
from fastapi import FastAPI, UploadFile, File

app = FastAPI(
    title="Audio Transcription Service",
    version="1.0.0"
)


@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "status": "audio_received"
    }